In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
train_data_path = 'DataFolder/train_masked.csv'
val_data_path   = 'DataFolder/val_masked.csv'
df_train = pd.read_csv(train_data_path)
df_val  = pd.read_csv(val_data_path)
df_train.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,1109,Well first you have to figure out if there is ...,No legal advice: Do not offer or request legal...,relationships,Perhaps offer him a plea deal of 20 minutes of...,"Firstly, the act of stealing the emails was il...",*that is more or less a binding agreement*\n\n...,But did he yell surprise? It's not rape if you...,1
1,490,cheap cigarettes online\nBuy Discount Duty Fre...,"No Advertising: Spam, referral links, unsolici...",pics,'m giving out Tyrande codes for 4 dollars upfr...,If anybody wants free case to open. Here is pr...,cauperheinea1970.tumblr.com - cam 2 cam now ch...,ヽ༼ ຈل͜ຈ༽ ﾉ Raise Them!\n\n ^^Dongers ^^Raised:...,1
2,159,"It's illegal, you can sue for pain and sufferi...",No legal advice: Do not offer or request legal...,legaladvice,Get a lawyer and get the security camera foota...,"Dear dumbass, they stole $1700 dollars from hi...",Can you beat and rape her? Then get her pregna...,Depends how much you want to keep your liquor ...,1
3,333,You should be fine. There have been cases wher...,No legal advice: Do not offer or request legal...,relationships,State dependant. There are states where it's p...,"make multiple deposits to multiple banks, stay...","Wait a few month, get her to some stairs and l...",But it's cheaper to just shoot them than to pe...,1
4,292,[Also Watch This Video - Olympics]([URL_dynami...,"No Advertising: Spam, referral links, unsolici...",videos,I just found that you can get 100 000 Pokemon ...,hunt for lady for jack off in neighbourhood [U...,SD Stream [English Stream]([URL_mntvlive]),"\nHey, you do not want to cmprar my awp asiimo...",0


In [3]:
import torch
from torch.utils.data import Dataset

class PairRedditRulesDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=384):
        self.labels = df["rule_violation"].astype(int).tolist()
        self.comment = df["body"].astype(str).tolist()
        self.context = (
            "RULE: " + df["rule"].astype(str) + "\n"
            "SUBREDDIT: " + df["subreddit"].astype(str) + "\n"
            "POS: " + df["positive_example_1"].fillna("").astype(str) + " || " +
                     df["positive_example_2"].fillna("").astype(str) + "\n"
            "NEG: " + df["negative_example_1"].fillna("").astype(str) + " || " +
                     df["negative_example_2"].fillna("").astype(str)
        ).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.comment[i],
            self.context[i],
            truncation="only_first",   # keep context intact
            max_length=self.max_len,
            padding=False,             # let collator handle padding
            return_tensors=None
        )
        
        enc["labels"] = int(self.labels[i])
        return enc


In [11]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader

bert_model = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(bert_model)
train_pairds = PairRedditRulesDataset(df_train, tokenizer)
val_pairds = PairRedditRulesDataset(df_val, tokenizer)
collate = DataCollatorWithPadding(tokenizer)

train_dataloader = DataLoader(train_pairds, batch_size=32, shuffle=True, collate_fn=collate)
val_dataloader = DataLoader(val_pairds, batch_size=32, shuffle=False, collate_fn=collate)

batch = next(iter(pairloader))

In [8]:
print(batch["input_ids"][1])
print(batch["input_ids"][0].dtype)

tensor([  101,  1031,  4773,  2069,  1033,  1031,  4748, 23467,  7929,  1033,
        17371,  1011,  2394,  1031,  5460,  1001,  1015,  1033,  1006,  1031,
        24471,  2140,  1035, 17249,  3207,  2595, 20205,  1033,  1007,   102,
         3627,  1024,  2053,  6475,  1024, 12403,  2213,  1010,  6523,  7941,
         6971,  1010,  4895, 19454, 28775,  3064,  6475,  1010,  1998, 10319,
         4180,  2024,  2025,  3039,  1012,  4942,  5596, 23194,  1024,  4715,
        21422,  2015, 13433,  2015,  1024,  7796,  1002,  3998,  1006,  2382,
         1010,  2199, 24471,  2685,  1007,  2044,  5938,  1002,  1017,  1010,
         2199,  2007,  1996,  5252, 10710,  5356,  1012,  1031, 24471,  2140,
         1035,  5252,  1033,  1064,  1064,  1045,  2031,  2048,  9537,  1010,
         2065,  2017,  2215,  2000,  5309,  1031, 24471,  2140,  1035,  1041,
         1011, 18015,  2666,  1033,  2065,  2017,  2215,  2000,  3477,  2007,
        18411,  2278,  4604,  6131, 11265,  2290,  1024,  1031, 

In [12]:
print(df_train.iloc[7])
print(train_pairds[7])
vocab = tokenizer.get_vocab()
print({k: v for k, v in vocab.items() if k in ['[CLS]', '[SEP]', '[PAD]', '[UNK]', '[MASK]']})

unused_count = 0
for k, v in vocab.items():
    if k.startswith('[unused'):
        unused_count += 1

print(f"Number of unused tokens in the tokenizer vocabulary: {unused_count} out of {len(vocab)} total tokens")

row_id                                                              892
body                  Are you Want online jobs....so our site help y...
rule                  No Advertising: Spam, referral links, unsolici...
subreddit                                                     worldnews
positive_example_1            watch good one hooters there [URL_https:]
positive_example_2    Earn $150 cash back with Chase Freedom®.  Appl...
negative_example_1    HD Streams: |[ENG HD Stoke vs Manchester Unite...
negative_example_2    HD Stream: [English stream]([URL_uclstreaming]...
rule_violation                                                        1
Name: 7, dtype: object
{'input_ids': [101, 2024, 2017, 2215, 3784, 5841, 1012, 1012, 1012, 1012, 2061, 2256, 2609, 2393, 2017, 1031, 24471, 2140, 1035, 23012, 11008, 1033, 102, 3627, 1024, 2053, 6475, 1024, 12403, 2213, 1010, 6523, 7941, 6971, 1010, 4895, 19454, 28775, 3064, 6475, 1010, 1998, 10319, 4180, 2024, 2025, 3039, 1012, 4942, 5596, 23194, 10

In [10]:
### Tokenizer test ###
test_text = "Hello I think this is an excellent way of doing this all || RULE: my rule || COMMENT: POS: NEG: SUBREDDIT: \n"
test_context = "RULE: hellooo"

enc = tokenizer(
            test_text,
            truncation=True,
            max_length=8,
            padding="max_length"
        )

pairenc = tokenizer(
            text = test_text,
            text_pair = test_context,
            truncation=True,
            max_length=40,
            padding="max_length"
        )
print(pairenc["input_ids"])
for id in pairenc["input_ids"]:
    for k, v in vocab.items():
        if v == id:
            print(f"{id}: {k}")
    

[101, 7592, 1045, 2228, 2023, 2003, 2019, 6581, 2126, 1997, 2725, 2023, 2035, 1064, 1064, 3627, 1024, 2026, 3627, 1064, 1064, 7615, 1024, 13433, 2015, 1024, 11265, 2290, 1024, 4942, 5596, 23194, 1024, 102, 3627, 1024, 7592, 9541, 102, 0]
101: [CLS]
7592: hello
1045: i
2228: think
2023: this
2003: is
2019: an
6581: excellent
2126: way
1997: of
2725: doing
2023: this
2035: all
1064: |
1064: |
3627: rule
1024: :
2026: my
3627: rule
1064: |
1064: |
7615: comment
1024: :
13433: po
2015: ##s
1024: :
11265: ne
2290: ##g
1024: :
4942: sub
5596: ##red
23194: ##dit
1024: :
102: [SEP]
3627: rule
1024: :
7592: hello
9541: ##oo
102: [SEP]
0: [PAD]


In [13]:
import torch

batch = next(iter(train_dataloader))
print({k: (v.shape, v.dtype) for k, v in batch.items()})

# Required keys:
# - input_ids:      [B, L], dtype long
# - attention_mask: [B, L], dtype long/bool
# - token_type_ids: [B, L] (ok if missing), dtype long
# - labels:         [B],    dtype long (0/1)

# If any dtype is wrong, quick fix before model call:
def to_device_and_fix(batch, device):
    out = {}
    for k, v in batch.items():
        if k in ("input_ids", "attention_mask", "token_type_ids"):
            out[k] = v.to(device).long()
        elif k == "labels":
            out[k] = v.to(device).long()
        else:
            out[k] = v.to(device)
    return out

device = "cuda" if torch.cuda.is_available() else "cpu"


{'input_ids': (torch.Size([32, 384]), torch.int64), 'token_type_ids': (torch.Size([32, 384]), torch.int64), 'attention_mask': (torch.Size([32, 384]), torch.int64), 'labels': (torch.Size([32]), torch.int64)}


In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
for p in model.base_model.parameters():
    p.requires_grad = False

for p in model.classifier.parameters():
    p.requires_grad = True
